# 03 · Checking the team's first findings

**Historical exploratory notebook.** The current shared-dataset workflow starts with `04_build_clean_data`, followed by `05_final_report` or `10`–`13`. This notebook uses earlier raw-data exploration and definitions; its outputs do not define the current cleaned tables.

Findings from the team's Document C, rerun on the data, raw and after cleaning.
Raw keeps cancelled orders in. Clean takes them out, and refunds are netted inside each order.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # run from the notebooks folder

import pandas as pd
from weavve import load, paths
from weavve.load import MARKETPLACE, WEBSITE
pd.set_option("display.max_columns", 50)
print("data:", paths.DATA)
print("outputs:", paths.OUT)
t = load.orders()
o = t.groupby("Order ID").agg(ch=("Order sales channel", "first"), net=("Net sales", "sum"), cancelled=("Is canceled order", "max"),
                              disc=("Discounts", "sum"), cust=("Customer ID", "first"))

**1. Orders and average order value by channel.**

In [ ]:
def split(d):
    g = d.groupby("ch").agg(orders=("net", "size"), sales=("net", "sum"))
    g["share_of_orders_%"] = (g.orders / g.orders.sum() * 100).round(2)
    g["share_of_sales_%"] = (g.sales / g.sales.sum() * 100).round(2)
    g["avg_order_$"] = (g.sales / g.orders).round(2)
    return g.loc[[MARKETPLACE, WEBSITE]]
display(split(o).assign(version="raw")); display(split(o[~o.cancelled]).assign(version="clean"))

**2. The biggest single sale line.**

In [ ]:
big_order = t.loc[t["Net sales"].idxmax(), "Order ID"]
t[t["Order ID"] == big_order][["Order ID", "Day", "Quantity ordered", "Net sales", "Order or sales reversal", "Is canceled order"]]

**3. Customers who bought only once**, by channel. Marketplace IDs rarely repeat, so read the website figure.

In [ ]:
c = o[~o.cancelled & o.cust.notna()]
once = {ch: round((c[c.ch == ch].groupby("cust").size() == 1).mean() * 100, 1) for ch in [WEBSITE, MARKETPLACE]}
once["all"] = round((c.groupby("cust").size() == 1).mean() * 100, 1)
once

**4. Orders with any discount.**

In [ ]:
clean = o[~o.cancelled]
{"raw": round((o.disc != 0).mean() * 100, 1), "clean": round((clean.disc != 0).mean() * 100, 1),
 "website": round((clean[clean.ch == WEBSITE].disc != 0).mean() * 100, 1), "marketplace": round((clean[clean.ch == MARKETPLACE].disc != 0).mean() * 100, 1)}

**5. Discount codes never used** (file 8).

In [ ]:
e = load.discounts()
round((e["Times Used In Total"].fillna(0) == 0).mean() * 100, 1)